In [ ]:
import os
import io
import json
import pickle
import boto3
import pandas as pd
import numpy as np
try:
    import xgboost as xgb
except:
    !pip install xgboost
    import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss, accuracy_score

#### Constants

In [ ]:
str_bucket = os.getcwd().split('/')[4].replace('_','-')
print(f'Bucket: {str_bucket}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_output = './output'
os.makedirs(str_output, exist_ok=True)

s3 = boto3.client('s3')

#### Functions

In [ ]:
def save_pickle_to_s3(obj, bucket, key):
    """Serialize a Python object and upload to S3."""
    buffer = io.BytesIO()
    pickle.dump(obj, buffer)
    buffer.seek(0)
    s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())
    print(f'Saved to s3://{bucket}/{key}')

#### Import Data from S3

In [ ]:
X_train = pd.read_csv(f's3://{str_bucket}/03_preprocessing/X_train.csv')
X_valid = pd.read_csv(f's3://{str_bucket}/03_preprocessing/X_valid.csv')
X_test = pd.read_csv(f's3://{str_bucket}/03_preprocessing/X_test.csv')

y_train = pd.read_csv(f's3://{str_bucket}/03_preprocessing/y_train.csv').squeeze()
y_valid = pd.read_csv(f's3://{str_bucket}/03_preprocessing/y_valid.csv').squeeze()
y_test = pd.read_csv(f's3://{str_bucket}/03_preprocessing/y_test.csv').squeeze()

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_valid: {X_valid.shape}, y_valid: {y_valid.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')

#### Encode Target Labels

In [ ]:
le = LabelEncoder()
le.fit(y_train)

y_train_enc = le.transform(y_train)
y_valid_enc = le.transform(y_valid)
y_test_enc = le.transform(y_test)

print(f'Classes: {le.classes_}')
print(f'Number of classes: {len(le.classes_)}')

#### Create DMatrices

In [ ]:
dtrain = xgb.DMatrix(X_train, label=y_train_enc)
dvalid = xgb.DMatrix(X_valid, label=y_valid_enc)
dtest = xgb.DMatrix(X_test, label=y_test_enc)

#### Train XGBoost Model

Multi-class classification with `multi:softprob` to get calibrated probability outputs for each pitch type.

In [ ]:
dict_params = {
    'objective': 'multi:softprob',
    'num_class': len(le.classes_),
    'eval_metric': 'mlogloss',
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'seed': 42,
    'verbosity': 1,
}

print('Parameters:')
for k, v in dict_params.items():
    print(f'  {k}: {v}')

In [ ]:
# train with early stopping on validation set
model = xgb.train(
    dict_params,
    dtrain,
    num_boost_round=1000,
    evals=[(dtrain, 'train'), (dvalid, 'valid')],
    early_stopping_rounds=50,
    verbose_eval=50,
)

print(f'\nBest iteration: {model.best_iteration}')
print(f'Best validation mlogloss: {model.best_score:.4f}')

#### Quick Validation Check

In [ ]:
# predictions on validation set
arr_valid_proba = model.predict(dvalid)
arr_valid_pred = le.classes_[arr_valid_proba.argmax(axis=1)]

flt_valid_acc = accuracy_score(y_valid, arr_valid_pred)
flt_valid_logloss = log_loss(y_valid_enc, arr_valid_proba)

print(f'Validation accuracy: {flt_valid_acc:.4f}')
print(f'Validation log-loss: {flt_valid_logloss:.4f}')

#### Save Model and Predictions

Model and prediction data go to S3. Only small param summary stays local.

In [ ]:
# save model to S3
str_model_local = '/tmp/xgb_model.json'
model.save_model(str_model_local)
s3.upload_file(str_model_local, str_bucket, f'{str_task}/xgb_model.json')
print(f'Saved model to s3://{str_bucket}/{str_task}/xgb_model.json')

# save label encoder to S3
save_pickle_to_s3(le, str_bucket, f'{str_task}/label_encoder.pkl')

# save test predictions to S3
arr_test_proba = model.predict(dtest)
arr_test_pred = le.classes_[arr_test_proba.argmax(axis=1)]

df_test_preds = pd.DataFrame(arr_test_proba, columns=[f'prob_{c}' for c in le.classes_])
df_test_preds['predicted'] = arr_test_pred
df_test_preds['actual'] = y_test.values
df_test_preds.to_csv(f's3://{str_bucket}/{str_task}/test_predictions.csv', index=False)
print(f'Saved test predictions to s3://{str_bucket}/{str_task}/test_predictions.csv')

# save training params locally (small file)
dict_params['best_iteration'] = model.best_iteration
dict_params['best_score'] = model.best_score
with open(f'{str_output}/params.json', 'w') as f:
    json.dump(dict_params, f, indent=2)
print(f'Saved params.json to {str_output}/')